# Cabra wearable: servos (PCA9685) + IMUs

Notebook para el firmware `ErisServoIMU` (flavor ErisServoDriver de Eris + 3 MPU6050 + IMU externo MPU6050, BMI160 o BNO055).

1. **Conexión**: abre el puerto, espera la calibración de los IMUs y activa el streaming `S_F IMU`.
2. **Trayectoria**: carga un CSV y lo convierte a ángulos de servo con `MAPEO`.
3. **Envío**: los puntos van a una cola en la ESP32, que los reproduce a periodo fijo (200 Hz).
4. **Gráfica en vivo**: ángulos de los IMUs y lo que se está mandando a los servos.
   Teclas en la ventana: `c` = nuevo cero de los IMUs y del IMU externo (pose actual = 0°), `t` = reproducir, `x` = detener, `i` = ir al inicio, `r` = todos los servos a 90°, `p` = seguir la fase del IMU externo, `o` = dejar de seguirla.
5. **Análisis**: IMUs medidos vs. trayectoria comandada, desde el CSV guardado.

Comandos del firmware: `X ch ang`, `Y ch ang` (suave), `T_CH`, `T_DT`, `P`, `T_GO`, `T_END`, `T_STOP`, `T_ST`, `CAL`, `IMU`, y para el IMU externo `PH_ON`, `PH_OFF`, `PH_CFG`, `PH_IMU`, `PH_W`, `PH_MODE`, `PH_GYRO`, `PH_ST`, `I2C_SCAN`.

In [17]:
%matplotlib qt
import re
import csv
import time
import struct
import threading
from collections import deque
import numpy as np
import pandas as pd
import serial
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# ---------------- Configuración ----------------
PUERTO = "COM9"
BAUDIOS = 921600                  # ERIS_SERIAL_BAUD del firmware en ESP32
ARTICULACIONES = ["hip", "knee", "ankle"]
ARCHIVO_CSV = f"imus_servos_{time.strftime('%Y%m%d_%H%M%S')}.csv"

# JointSample_t del firmware: t_ms, hip, knee, ankle, traj_idx, traj_cola
MUESTRA = struct.Struct("<6f")
# ExtSample_t (feature EXT): t_ms, theta, omega, alpha, fase, hip, knee, ankle
MUESTRA_EXT = struct.Struct("<8f")


def cobs_decode(datos):
    """Decodifica una trama COBS (PacketSerial) sin el 0x00 final."""
    salida = bytearray()
    i, n = 0, len(datos)
    while i < n:
        codigo = datos[i]
        if codigo == 0 or i + codigo > n:
            raise ValueError("trama COBS inválida")
        salida += datos[i + 1:i + codigo]
        i += codigo
        if codigo < 0xFF and i < n:
            salida.append(0)
    return bytes(salida)


# Líneas de texto (Serial.println del firmware) que llegan mezcladas con las tramas binarias
_TEXTO = re.compile(rb"(?:[\t\x20-\x7e]+\r?\n)+")


class Eris:
    """Conexión con el firmware ErisServoIMU.

    Un hilo lee el puerto sin parar: separa las tramas COBS (terminadas en 0x00)
    del texto plano, guarda las muestras de los IMUs en `muestras` y en el CSV,
    y deja los mensajes de texto en `mensajes`.
    """

    def __init__(self, puerto, baudios, archivo_csv, eco=True):
        self.ser = serial.Serial(puerto, baudios, timeout=0.05)
        self.eco = eco
        self.lock = threading.Lock()          # protege muestras / estado
        self.lock_tx = threading.Lock()       # un solo escritor a la vez en el puerto
        self.muestras = deque(maxlen=500_000) # (t_s, hip, knee, ankle, idx, cola)
        self.ext = deque(maxlen=500_000)      # (t_s, theta, omega, alpha, fase, hip, knee, ankle)
        self.mensajes = deque(maxlen=500)     # (n, texto)
        self.n_mensajes = 0
        self.ultimo_idx = -1                  # puntos reproducidos (-1 = sin trayectoria)
        self.ultima_cola = 0
        self.tramas_malas = 0
        self.tray_enviada = None              # array (N, k) que se está reproduciendo
        self._buf = b""
        self._corriendo = True

        self.archivo = open(archivo_csv, "w", newline="")
        self.escritor = csv.writer(self.archivo)
        self.escritor.writerow(["t_s"] + ARTICULACIONES + ["traj_idx", "traj_cola"])

        self.hilo = threading.Thread(target=self._leer, daemon=True)
        self.hilo.start()

    # ---------- envío ----------
    def enviar(self, comando):
        self.enviar_lineas([comando])

    def enviar_lineas(self, lineas):
        with self.lock_tx:
            self.ser.write(("\n".join(lineas) + "\n").encode())

    def marca(self):
        """Número del próximo mensaje; tómala ANTES de enviar el comando que esperas."""
        return self.n_mensajes

    def esperar_texto(self, fragmento, desde, timeout=5.0):
        """Espera un mensaje (número >= desde) que contenga `fragmento`. True si llegó."""
        t_fin = time.time() + timeout
        while time.time() < t_fin:
            with self.lock:
                if any(n >= desde and fragmento in m for n, m in self.mensajes):
                    return True
            time.sleep(0.02)
        return False

    # ---------- recepción ----------
    def _texto(self, crudo):
        for linea in crudo.decode(errors="replace").splitlines():
            linea = linea.strip()
            if linea:
                with self.lock:
                    self.mensajes.append((self.n_mensajes, linea))
                    self.n_mensajes += 1
                if self.eco:
                    print("<-", linea)

    def _paquete(self, trama):
        """Intenta interpretar la trama como paquete Eris. True si era válido."""
        try:
            d = cobs_decode(trama)
        except ValueError:
            return False
        tipo = d[:1]
        if tipo == b"D" and len(d) >= 2:
            # Una sección por feature, en el orden de "S_F IMU EXT": [n][n muestras]
            secciones, pos = [], 1
            for formato in (MUESTRA, MUESTRA_EXT):
                if pos >= len(d):
                    return False
                n = d[pos]
                fin = pos + 1 + n * formato.size
                if fin > len(d):
                    return False
                secciones.append([formato.unpack_from(d, pos + 1 + k * formato.size) for k in range(n)])
                pos = fin
            if pos != len(d):
                return False
            filas, filas_ext = secciones
            with self.lock:
                for t_ms, h, c, ca, idx, cola in filas:
                    self.muestras.append((t_ms / 1000, h, c, ca, idx, cola))
                    self.escritor.writerow([f"{t_ms / 1000:.4f}", f"{h:.2f}", f"{c:.2f}",
                                            f"{ca:.2f}", int(idx), int(cola)])
                if filas:
                    self.ultimo_idx = int(filas[-1][4])
                    self.ultima_cola = int(filas[-1][5])
                for t_ms, th, om, al, fase, h, c, ca in filas_ext:
                    self.ext.append((t_ms / 1000, th, om, al, fase, h, c, ca))
            return True
        if tipo in (b"T", b"E"):
            self._texto((b"ERROR: " if tipo == b"E" else b"") + d[1:].replace(b"\0", b" ") + b"\n")
            return True
        return False

    def _trama(self, trama):
        if not trama or self._paquete(trama):
            return
        # Texto pegado delante de un paquete (el firmware imprime texto plano con Serial.println)
        m = _TEXTO.match(trama)
        if m:
            self._texto(m.group())
            resto = trama[m.end():]
            if not resto or self._paquete(resto):
                return
        self.tramas_malas += 1

    def _procesar(self, datos):
        self._buf += datos
        *tramas, self._buf = self._buf.split(b"\0")
        for trama in tramas:
            self._trama(trama)
        # Texto completo sin paquete detrás (p. ej. respuestas con el streaming apagado)
        if self._buf.endswith(b"\n") and _TEXTO.fullmatch(self._buf):
            self._texto(self._buf)
            self._buf = b""

    def _leer(self):
        t_flush = time.time()
        while self._corriendo:
            try:
                datos = self.ser.read(max(1, self.ser.in_waiting))
            except serial.SerialException:
                break
            if datos:
                self._procesar(datos)
            if time.time() - t_flush > 1:
                self.flush()
                t_flush = time.time()

    def flush(self):
        with self.lock:
            self.archivo.flush()

    def cerrar(self):
        self._corriendo = False
        self.hilo.join(timeout=1)
        try:
            self.enviar("T_STOP")
            self.enviar("S_OFF")
        except serial.SerialException:
            pass
        self.ser.close()
        self.archivo.close()


# ---------------- Conexión ----------------
try:
    eris.cerrar()        # si re-ejecutas la celda, cierra la conexión anterior
except NameError:
    pass

eris = Eris(PUERTO, BAUDIOS, ARCHIVO_CSV)
print(f"Conectado a {PUERTO}. La ESP32 se reinicia y calibra: deja la pata QUIETA...")
if not eris.esperar_texto("Serial Commands are ready", desde=0, timeout=12):
    print("(no llegó el mensaje de arranque; continúo de todos modos)")

eris.enviar("S_F IMU EXT")
eris.enviar("S_TIME")
eris.enviar("S_ON")
time.sleep(0.5)
print(f"Streaming activo, {len(eris.muestras)} muestras de los IMUs y {len(eris.ext)} del IMU externo. "
      f"Guardando los IMUs en {ARCHIVO_CSV}")
if not eris.ext:
    print("AVISO: sin datos del IMU externo (MPU6050, BMI160 o BNO055); revisa el cableado con eris.enviar('I2C_SCAN')")

Conectado a COM9. La ESP32 se reinicia y calibra: deja la pata QUIETA...
(no llegó el mensaje de arranque; continúo de todos modos)
<- Function requested: IMU
<- IMU selected
<- Function requested: EXT
<- EXT selected
<- Features Ready
Streaming activo, 112 muestras de los IMUs y 112 del IMU externo. Guardando los IMUs en imus_servos_20261008_192956.csv


## Prueba del IMU externo (MPU6050, BMI160 o BNO055)

Comprueba el IMU externo con el firmware ya subido, sin tocar el `.ino`: escaneo I2C de los otros buses, estado y datos en vivo. El sensor va en SDA 25 y SCL 26 y el firmware detecta cuál es.
Ejecútala después de la celda de conexión. Mientras dura la prueba el seguimiento de fase queda apagado,
así que los servos no se mueven. `probar_imu_externo(segundos)` se puede repetir las veces que quieras.

In [117]:
def probar_imu_externo(segundos=6):
    """Prueba el IMU externo (MPU6050, BMI160 o BNO055) con el firmware ya subido: escaneo I2C, estado y datos en vivo."""
    eris.enviar("PH_OFF")                  # que los servos no se muevan durante la prueba
    marca = eris.marca()
    eris.enviar("I2C_SCAN")
    eris.enviar("PH_ST")
    time.sleep(1.0)
    with eris.lock:
        texto = [m for n, m in eris.mensajes if n >= marca]

    # 1. Escaneo I2C de los buses por hardware (el BNO055 va en su propio bus por software y no aparece aquí)
    wire = next((m for m in texto if m.startswith("I2C Wire ")), None)
    if wire is None:
        print("1. FALLA: no llegó la línea de I2C_SCAN. Si PH_ST (punto 2) sí responde, vuelve a subir el firmware:")
        print("   la versión anterior imprimía el escaneo en pedazos y el streaming lo rompía.")
    else:
        dirs = {int(a, 16) for a in re.findall(r"0x([0-9A-Fa-f]+)", wire)}
        print(f"1. Direcciones en el bus Wire (PCA9685 + hip): {sorted(hex(d) for d in dirs)}")
        print("   PCA9685 (0x40):", "OK" if 0x40 in dirs else "NO responde")
        print("   MPU6050 del hip (0x68/0x69):", "OK" if dirs & {0x68, 0x69} else "NO responde")

    # 2. Estado según el firmware
    st = next((m for m in texto if m.startswith("PH_ST")), None)
    if st is None:
        print("2. FALLA: el firmware no respondió a PH_ST (versión sin IMU externo).")
    elif "present=1" in st:
        cal = re.search(r"cal=(\d)(\d)(\d)(\d)", st)
        m_sensor = re.search(r"sensor=(\w+)", st)
        sensor = m_sensor.group(1) if m_sensor else "IMU externo"   # firmware anterior: sin campo sensor=
        print(f"2. El firmware encontró el {sensor} (SDA 25 / SCL 26):", re.search(r"addr=0x[0-9A-Fa-f]+", st).group())
        if sensor in ("BMI160", "MPU6050"):
            print(f"   El {sensor} calibra su giroscopio al encender: si lo moviste entonces, reinicia la ESP32 con el sensor quieto.")
        elif cal:
            sys_, gyr, acc, mag = map(int, cal.groups())
            print(f"   Calibración (0 a 3): sistema {sys_}, giroscopio {gyr}, acelerómetro {acc}")
            if gyr < 3:
                print("   El giroscopio aún no está calibrado: deja el sensor quieto unos segundos y repite la prueba.")
    else:
        print("2. FALLA: el firmware no encontró el MPU6050, el BMI160 ni el BNO055 al arrancar (present=0).")
        print("   Revisa VIN y GND y SDA a GPIO 25 y SCL a GPIO 26 (no 21/22).")
        print("   MPU6050: AD0 a GND (0x68) o 3V3 (0x69). BMI160: CS a 3V3 (modo I2C) y SAO a GND o 3V3. BNO055: PS0 y PS1 a GND.")
        print("   Corrige el cableado y reinicia la ESP32: solo lo busca al encender.")

    # Lecturas I2C del IMU externo desde que arrancó la ESP32
    if st is not None:
        m_rd = re.search(r"reads=(\d+) fails=(\d+)", st)
        if m_rd:
            buenas, malas = int(m_rd.group(1)), int(m_rd.group(2))
            total = buenas + malas
            print(f"   Lecturas I2C: {buenas} buenas y {malas} fallidas")
            if total and malas / total > 0.01:
                print("   AVISO: el bus lee mal. Causa probable: pull-ups débiles. Conecta una resistencia de 4.7 kohm de SDA a 3V3 y otra de SCL a 3V3,")
                print("          o sube EXT_I2C_HALF_PERIOD_US a 5 en configuration.h y vuelve a subir el firmware.")

    # 3. Datos en vivo
    with eris.lock:
        n0 = len(eris.ext)
    print(f"3. Mueve e inclina el sensor durante {segundos} s...")
    time.sleep(segundos)
    with eris.lock:
        filas = np.array(list(eris.ext)[n0:])
    if len(filas) == 0:
        print("   FALLA: no llegan datos del IMU externo (feature EXT).")
        print("   Si el punto 2 dice present=0, es el cableado; si no, vuelve a ejecutar la celda de conexión.")
        return
    t, theta, omega = filas[:, 0], filas[:, 1], filas[:, 2]
    tasa = (len(filas) - 1) / (t[-1] - t[0]) if len(filas) > 1 else 0
    print(f"   {len(filas)} muestras, {tasa:.0f} Hz (esperado ~100 Hz)")
    print(f"   pitch: {theta.min():.1f}° a {theta.max():.1f}° (rango {np.ptp(theta):.1f}°) | velocidad máx. {np.abs(omega).max():.0f} °/s")
    if np.ptp(theta) < 2 and np.abs(omega).max() < 5:
        print("   AVISO: casi no cambia. Para esta prueba hay que girar la placa alrededor del eje de giro configurado,")
        print("          El pitch es el giro alrededor del eje configurado en EXT_AXIS_W (ahora X): gira la placa alrededor de ese eje.")
        print("          Si giras alrededor de otro eje, la gravedad no cambia en el plano del pitch y no se mueve.")
        marca2 = eris.marca()
        eris.enviar("PH_ST")
        time.sleep(0.5)
        with eris.lock:
            st2 = next((m for n, m in eris.mensajes if n >= marca2 and m.startswith("PH_ST")), "")
        g = re.search(r"grv=([-\d.]+),([-\d.]+),([-\d.]+)", st2)
        w = re.search(r"gyr=([-\d.]+),([-\d.]+),([-\d.]+)", st2)
        if g:
            gx, gy, gz = map(float, g.groups())
            print(f"          Gravedad leída (m/s²): x={gx:.2f} y={gy:.2f} z={gz:.2f} | giro (°/s): {w.group(1)}, {w.group(2)}, {w.group(3)}")
            if abs(gx) < 1 and abs(gy) < 1 and abs(gz) > 8:
                print("          -> la placa está plana (Z vertical): por eso el pitch no cambia. Ponla de canto y gírala en su plano.")
            elif max(abs(gx), abs(gy), abs(gz)) < 0.05:
                print("          -> el sensor entrega todo en cero: no está haciendo la fusión. Revisa la alimentación y reinicia la ESP32.")
    else:
        # La velocidad del giroscopio debe seguir la derivada del pitch; si van en contra, el eje o el signo no coinciden
        r = np.corrcoef(np.gradient(theta, t), omega)[0, 1]
        print(f"   correlación entre velocidad y derivada del pitch: {r:+.2f}")
        if r < -0.3:
            print("   AVISO: la velocidad tiene el signo contrario. Cambia EXT_GYRO_SIGN a -1 en configuration.h y vuelve a subir el firmware,")
            print("          o revisa EXT_AXIS_W: el eje de giro debe ser perpendicular al plano en que mueves el sensor.")
        elif r < 0.5:
            print("   AVISO: la velocidad no concuerda bien con el pitch. Inclina el sensor en un solo plano (como el pitch de la pata).")
        else:
            print("   OK: el IMU externo responde al movimiento.")

    fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
    a1.plot(t - t[0], theta)
    a1.set_ylabel("Pitch (°)")
    a2.plot(t - t[0], omega, color="tab:orange")
    a2.set_ylabel("Velocidad (°/s)")
    a2.set_xlabel("Tiempo (s)")
    for ax in (a1, a2):
        ax.grid(True, alpha=0.3)
    fig.suptitle("Prueba del IMU externo")
    plt.show()


probar_bno055 = probar_bmi160 = probar_imu_externo      # nombres anteriores
probar_imu_externo()

<- PH_OFF ok
<- I2C Wire  (SDA 21, SCL 22: PCA9685 + hip): 0x40 0x68
<- I2C Wire1 (SDA 18, SCL 19: knee): 0x68
<- PH_ST present=1 sensor=BMI160 addr=0x68 cal=0000 track=0 theta=-0.0 omega=-0.1 alpha=-2 phase=0.421 hip=50.4 knee=121.5 ankle=114.9 grv=-0.87,-0.18,-0.54 gyr=-0.5,-0.2,0.4 reads=377 fails=0
1. Direcciones en el bus Wire (PCA9685 + hip): ['0x40', '0x68']
   PCA9685 (0x40): OK
   MPU6050 del hip (0x68/0x69): OK
2. El firmware encontró el BMI160 (SDA 25 / SCL 26): addr=0x68
   El BMI160 calibra su giroscopio al encender: si lo moviste entonces, reinicia la ESP32 con el sensor quieto.
   Lecturas I2C: 377 buenas y 0 fallidas
3. Mueve e inclina el sensor durante 6 s...
   601 muestras, 100 Hz (esperado ~100 Hz)
   pitch: -1.0° a 101.5° (rango 102.5°) | velocidad máx. 276 °/s
   correlación entre velocidad y derivada del pitch: +0.98
   OK: el IMU externo responde al movimiento.


## Trayectoria

Solo se usan las patas delanteras: `fr` (derecha) o `fl` (izquierda), que es la misma curva
desfasada medio ciclo. Hip, knee y ankle van a los canales 0, 1 y 2 de la PCA9685.

`servo = offset + signo * ángulo_del_CSV`, recortado a 0–180°. Ajusta `signo` y `offset` a cómo
montaste cada servo.

In [19]:
ARCHIVO_TRAY = "leg_actuator_angles_one_cycle.csv"   # o "leg_actuator_angles_trajectory_adapted.csv"
REPETICIONES = 10                                    # veces que se repite el archivo

PIERNA = "fr"                                        # "fr" o "fl"

# articulación: (canal PCA9685, signo, offset °)
MAPEO = {
    "hip":   (0, +1, 0.0),
    "knee":  (1, +1, 0.0),
    "ankle": (2, +1, 0.0),
}

tray = pd.read_csv(ARCHIVO_TRAY)
DT_S = float(np.median(np.diff(tray["time_s"])))
columnas = [f"{PIERNA}_{a}_joint" for a in MAPEO]
canales = [MAPEO[a][0] for a in MAPEO]
signo = np.array([MAPEO[a][1] for a in MAPEO], dtype=float)
offset = np.array([MAPEO[a][2] for a in MAPEO], dtype=float)

servo = offset + signo * tray[columnas].to_numpy()
for j, c in enumerate(columnas):
    fuera = np.mean((servo[:, j] < 0) | (servo[:, j] > 180))
    if fuera > 0:
        print(f"AVISO: {c} (canal {canales[j]}) sale de 0-180° en el {fuera:.0%} de los puntos; se recorta")
servo = np.clip(servo, 0, 180)
servo_env = np.tile(servo, (REPETICIONES, 1))

print(f"{ARCHIVO_TRAY}: {len(tray)} puntos, dt = {DT_S * 1000:.1f} ms, "
      f"{len(servo_env)} puntos a enviar ({len(servo_env) * DT_S:.1f} s)")

fig, axs = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
t = tray["time_s"].to_numpy()
for j, c in enumerate(columnas):
    axs[0].plot(t, tray[c], label=c)
    axs[1].plot(t, servo[:, j], label=f"{list(MAPEO)[j]} (ch {canales[j]})")
axs[0].set_ylabel("Ángulo articular (°)")
axs[1].set_ylabel("Ángulo de servo (°)")
axs[1].set_xlabel("Tiempo (s)")
axs[0].legend(fontsize=8)
axs[1].legend(fontsize=8)
for ax in axs:
    ax.grid(True, alpha=0.3)
fig.suptitle(f"Una repetición de la trayectoria ({PIERNA})")
plt.show()

leg_actuator_angles_one_cycle.csv: 127 puntos, dt = 5.0 ms, 1270 puntos a enviar (6.4 s)


## Envío de la trayectoria

El envío corre en un hilo aparte para que la gráfica en vivo siga funcionando.
Se mantienen `ADELANTO` puntos en la cola de la ESP32 (la cola usa lo que reporta el propio streaming),
así que el ritmo lo marca el reloj de la ESP32 y no el de Windows.

In [21]:
ADELANTO = 200          # puntos en cola en la ESP32 (1 s a 200 Hz; la cola admite 511)
MAX_LINEA = 64          # SERIALCOMMAND_BUFFER del firmware

_envio = {"hilo": None, "parar": False}
_fase = {"on": False, "intervalos": []}   # seguimiento de fase: estado e intervalos [t_ini, t_fin] (reloj de la ESP32)


def _marcar_fase(on):
    """Registra cuándo se enciende o apaga el seguimiento, para dibujar los comandos que mandó."""
    if on == _fase["on"]:
        return
    with eris.lock:
        t = eris.ext[-1][0] if eris.ext else 0.0
    _fase["on"] = on
    if on:
        _fase["intervalos"].append([t, None])
    else:
        _fase["intervalos"][-1][1] = t


def linea_punto(fila):
    return "P " + " ".join(str(int(round(v * 10))) for v in fila)   # décimas de grado


def ir_a_inicio(esperar=2.0):
    """Lleva cada servo suavemente (comando Y) al primer punto de la trayectoria."""
    eris.enviar_lineas([f"Y {ch} {v:.1f}" for ch, v in zip(canales, servo_env[0])])
    time.sleep(esperar)


def _enviar_trayectoria():
    lineas = [linea_punto(f) for f in servo_env]
    largo = max(map(len, lineas))
    if largo > MAX_LINEA:
        print(f"ERROR: las líneas P miden {largo} caracteres (máx. {MAX_LINEA}); usa menos columnas")
        return

    marca = eris.marca()
    eris.enviar_lineas(["T_STOP", f"T_DT {DT_S * 1000:g}", "T_CH " + " ".join(map(str, canales))])
    if not eris.esperar_texto("T_CH ok", marca, timeout=2):
        print("ERROR: el firmware no confirmó T_CH")
        return
    with eris.lock:
        eris.ultimo_idx = -1
    eris.tray_enviada = servo_env

    n_total = len(lineas)
    enviados = min(ADELANTO, n_total)
    eris.enviar_lineas(lineas[:enviados])      # precarga
    eris.enviar("T_GO")
    t_inicio = time.time()

    while enviados < n_total and not _envio["parar"]:
        reproducidos = max(eris.ultimo_idx, 0)
        falta = ADELANTO - (enviados - reproducidos)
        if falta > 0:
            bloque = lineas[enviados:enviados + falta]
            eris.enviar_lineas(bloque)
            enviados += len(bloque)
        time.sleep(0.01)

    if _envio["parar"]:
        eris.enviar("T_STOP")
        print(f"Trayectoria detenida en el punto {eris.ultimo_idx} de {n_total}")
        return

    marca = eris.marca()
    eris.enviar("T_END")
    restante = (n_total - max(eris.ultimo_idx, 0)) * DT_S
    if not eris.esperar_texto("Trayectoria terminada", marca, timeout=restante + 5):
        print("AVISO: no llegó 'Trayectoria terminada'")
    print(f"Trayectoria completa: {n_total} puntos en {time.time() - t_inicio:.2f} s "
          f"(esperado {n_total * DT_S:.2f} s)")
    eris.enviar("T_ST")     # underruns / rechazados


def reproducir(desde_inicio=True):
    """Arranca la trayectoria en segundo plano."""
    if _envio["hilo"] is not None and _envio["hilo"].is_alive():
        print("Ya hay una trayectoria en curso (usa detener())")
        return
    _marcar_fase(False)       # T_GO apaga el seguimiento en la ESP32

    def tarea():
        if desde_inicio:
            ir_a_inicio()
        _enviar_trayectoria()

    _envio["parar"] = False
    _envio["hilo"] = threading.Thread(target=tarea, daemon=True)
    _envio["hilo"].start()


def detener():
    _envio["parar"] = True
    _marcar_fase(False)
    eris.enviar("T_STOP")
    eris.enviar("PH_OFF")


def a_90():
    """Detiene la trayectoria y lleva todos los servos suavemente a 90°."""
    detener()
    eris.enviar_lineas([f"Y {ch} 90" for ch in canales])

## Seguimiento de fase (IMU externo)

El IMU externo (MPU6050, BMI160 o BNO055) va pegado al cuerpo. La ESP32 estima el pitch, la velocidad y la aceleración, busca la fase
de la marcha en la tabla (`phase_table.h`, generada con `tools/gen_phase_table.py`) y manda ese hip, knee y
ankle a los servos con el mismo `MAPEO` de la trayectoria. La búsqueda corre en la ESP32 a 100 Hz.

- `fase_on()` / `fase_off()` (teclas `p` / `o` en la gráfica). Encender el seguimiento detiene la trayectoria y al revés.
- Si al mover el sensor la fase retrocede o se pierde, prueba `IMU_SIGNO = -1`.
- El pitch que ves vale **0° en la pose en que calibras** el IMU externo (al encender, o con la tecla `c`). La tabla
  recibe ese pitch más `IMU_OFFSET`, el pitch del pie en esa pose (por defecto el medio de la tabla).
- `alinear_offset()`: con la persona caminando a ritmo normal, ajusta `IMU_OFFSET` para que el pitch medio
  del sensor coincida con el pitch medio del pie de la tabla (el montaje en el cuerpo no coincide con el del CSV).
- El pitch es el giro alrededor del eje X de la placa (`EXT_AXIS_W` en `configuration.h`), que debe ser perpendicular al plano en que se mueve el segmento. Un desfase constante lo corrige la calibración con `c` y `alinear_offset()`.

In [22]:
IMU_SIGNO = -1            # sentido del pitch: -1 lo invierte (igual que EXT_PITCH_SIGN del firmware); +1 si crece como el del pie
PESOS = (1, 1, 0)         # theta, omega, alpha: la mejor configuración del prototipo
MODO = 2                  # 0 = ventana, 1 = global, 2 = reenganche (recover)

# Pitch medio del pie en la tabla (hip + knee + ankle - 180), para alinear_offset()
THETA_TABLA = float(tray[[f"{PIERNA}_{j}_joint" for j in ("hip", "knee", "ankle")]].sum(axis=1).mean() - 180)

# Pitch del pie (según la tabla) en la pose en la que calibras el IMU externo con `c`. El pitch que ves vale 0° en esa
# pose; la tabla recibe ese pitch + IMU_OFFSET. Por defecto, el pitch medio del pie; alinear_offset() lo ajusta.
IMU_OFFSET = THETA_TABLA


def configurar_fase():
    """Manda a la ESP32 el mapeo a servos, el montaje del IMU y los parámetros de la búsqueda."""
    cfg = []
    for ch, sg, off in MAPEO.values():
        cfg += [ch, sg, round(off * 10)]
    eris.enviar_lineas([
        "PH_CFG " + " ".join(map(str, cfg)),
        f"PH_IMU {IMU_SIGNO} {IMU_OFFSET:.2f}",
        "PH_W " + " ".join(map(str, PESOS)),
        f"PH_MODE {MODO}",
    ])


def fase_on():
    configurar_fase()
    eris.enviar("PH_ON")
    _marcar_fase(True)


def fase_off():
    eris.enviar("PH_OFF")
    _marcar_fase(False)


def alinear_offset(segundos=10):
    """Mide `segundos` de marcha y ajusta IMU_OFFSET para que el pitch medio coincida con la tabla."""
    global IMU_OFFSET
    print(f"Camina a ritmo normal durante {segundos} s...")
    with eris.lock:
        n0 = len(eris.ext)
    time.sleep(segundos)
    with eris.lock:
        theta = np.array([m[1] for m in list(eris.ext)[n0:]])
    if len(theta) < 10:
        print("No llegaron datos del IMU externo")
        return
    IMU_OFFSET = THETA_TABLA - theta.mean()      # theta es relativo a la pose calibrada (0° allí)
    print(f"Pitch medio {theta.mean():.1f}° respecto a la pose calibrada (tabla {THETA_TABLA:.1f}°) -> IMU_OFFSET = {IMU_OFFSET:.1f}°")
    eris.enviar(f"PH_IMU {IMU_SIGNO} {IMU_OFFSET:.2f}")


configurar_fase()
eris.enviar("PH_ST")

<- PH_CFG ok
<- PH_IMU ok
<- PH_W ok
<- PH_MODE ok
<- PH_ST present=1 sensor=MPU6050 addr=0x68 cal=0000 track=0 theta=-0.4 omega=-0.6 alpha=-3 phase=0.421 hip=50.4 knee=121.5 ankle=114.9 grv=-0.77,-0.63,-0.00 gyr=-4.7,1.8,-1.2 reads=245104 fails=0


## Gráfica en vivo

Arriba: ángulos de los IMUs. En medio: ángulo que la ESP32 está mandando a cada servo. Abajo: pitch del IMU externo y fase estimada.
Teclas con la ventana activa: `c` cero de los IMUs y del IMU externo, pose actual = 0° (dejar la pata y el sensor quietos ~2 s), `t` reproducir, `x` detener, `i` ir al inicio, `r` todos los servos a 90°, `p` seguir la fase del IMU externo, `o` dejar de seguirla.

In [ ]:
VENTANA_S = 10

fig, (ax_imu, ax_srv, ax_ext) = plt.subplots(3, 1, sharex=True, figsize=(11, 9))
lineas_imu = {a: ax_imu.plot([], [], label=a)[0] for a in ARTICULACIONES}
lineas_srv = [ax_srv.plot([], [], label=f"{a} (ch {ch})")[0] for a, ch in zip(MAPEO, canales)]
# Comandos del seguimiento de fase (línea punteada, mismo color que su servo)
lineas_srv_fase = [ax_srv.plot([], [], "--", lw=2, color=l.get_color(), label=f"{a} (fase)")[0]
                   for l, a in zip(lineas_srv, MAPEO)]
ax_imu.set_ylabel("Ángulo IMU (°)")
ax_srv.set_ylabel("Comando servo (°)")
ax_ext.set_xlabel("Tiempo ESP32 (s)")
ax_ext.set_ylabel("Pitch IMU externo (°)")
linea_theta = ax_ext.plot([], [], color="tab:purple", label="pitch IMU externo")[0]
ax_fase = ax_ext.twinx()
ax_fase.set_ylabel("Fase estimada (ciclos)")
ax_fase.set_ylim(0, 1)
linea_fase = ax_fase.plot([], [], color="tab:orange", lw=1, label="fase estimada")[0]
ax_ext.legend(handles=[linea_theta, linea_fase], loc="upper left")
ax_srv.set_ylim(-5, 185)
ax_imu.legend(loc="upper left")
ax_srv.legend(loc="upper left")
for ax in (ax_imu, ax_srv, ax_ext):
    ax.grid(True, alpha=0.3)
titulo = fig.suptitle("")


def ventana(muestras=None):
    """Muestras de los últimos VENTANA_S segundos (más antiguas primero)."""
    with eris.lock:
        muestras = eris.muestras if muestras is None else muestras
        if not muestras:
            return None
        t_fin = muestras[-1][0]
        recientes = []
        for m in reversed(muestras):
            if m[0] < t_fin - VENTANA_S:
                break
            recientes.append(m)
    return np.array(recientes[::-1])


def actualizar(_):
    datos = ventana()
    if datos is None or len(datos) < 2:
        return []
    t = datos[:, 0]
    for k, a in enumerate(ARTICULACIONES):
        lineas_imu[a].set_data(t, datos[:, 1 + k])

    # punto aplicado = idx - 1 dentro de la trayectoria enviada
    idx = datos[:, 4].astype(int) - 1
    tray_env = eris.tray_enviada
    for j, linea in enumerate(lineas_srv):
        if tray_env is not None and j < tray_env.shape[1]:
            valido = (idx >= 0) & (idx < len(tray_env))
            y = np.full(len(t), np.nan)
            y[valido] = tray_env[idx[valido], j]
            linea.set_data(t, y)

    ax_imu.set_xlim(max(0, t[-1] - VENTANA_S), max(VENTANA_S, t[-1]))
    ang = datos[:, 1:4]
    ax_imu.set_ylim(ang.min() - 5, ang.max() + 5)

    ext = ventana(eris.ext)
    if ext is not None and len(ext) >= 2:
        linea_theta.set_data(ext[:, 0], ext[:, 1])
        linea_fase.set_data(ext[:, 0], ext[:, 4])
        ax_ext.set_ylim(ext[:, 1].min() - 5, ext[:, 1].max() + 5)

        # Comando de cada servo = offset + signo * articulación elegida, solo mientras el seguimiento estuvo activo
        activo = np.zeros(len(ext), bool)
        for t_ini, t_fin in _fase["intervalos"]:
            activo |= (ext[:, 0] >= t_ini) & (ext[:, 0] <= (np.inf if t_fin is None else t_fin))
        for j, (a, linea) in enumerate(zip(MAPEO, lineas_srv_fase)):
            _, sg, off = MAPEO[a]
            y = np.clip(off + sg * ext[:, 5 + j], 0, 180)
            y[~activo] = np.nan
            linea.set_data(ext[:, 0], y)

    estado = f"trayectoria: punto {eris.ultimo_idx}" if eris.ultimo_idx >= 0 else "trayectoria: parada"
    seguimiento = "seguimiento de fase: ON" if _fase["on"] else "seguimiento de fase: off"
    titulo.set_text(f"{estado} | {seguimiento} | cola ESP32: {eris.ultima_cola} | tramas malas: {eris.tramas_malas}")
    return list(lineas_imu.values()) + lineas_srv + lineas_srv_fase + [linea_theta, linea_fase]


def al_teclear(evento):
    if evento.key == "c":
        eris.enviar("CAL")          # pose actual = cero de los MPU6050
        eris.enviar("PH_GYRO")      # recalibra el giroscopio del IMU externo (quieto ~2 s; en pausa mientras tanto)
        eris.enviar("PH_ZERO")      # y toma su pose actual como 0° del pitch
    elif evento.key == "t":
        reproducir()
    elif evento.key == "x":
        detener()
    elif evento.key in ("r", "R"):
        a_90()
    elif evento.key == "i":
        threading.Thread(target=ir_a_inicio, daemon=True).start()
    elif evento.key == "p":
        fase_on()
    elif evento.key == "o":
        fase_off()


for tecla in "ctxirpo":        # quita los atajos de matplotlib que usan estas teclas
    for nombre in ("keymap.back", "keymap.forward", "keymap.home", "keymap.pan", "keymap.zoom"):
        if tecla in plt.rcParams[nombre]:
            plt.rcParams[nombre].remove(tecla)
fig.canvas.mpl_connect("key_press_event", al_teclear)
anim = FuncAnimation(fig, actualizar, interval=40, cache_frame_data=False)
plt.show()

<- PH_ZERO ok
<- Calibrando giroscopio externo, no muevas el IMU...
<- Calibrando cero, no muevas la pata...
<- Cero listo.
<- Giroscopio externo listo.
<- PH_ZERO ok
<- Calibrando cero, no muevas la pata...
<- Calibrando giroscopio externo, no muevas el IMU...
<- Cero listo.
<- Giroscopio externo listo.
<- PH_CFG ok
<- PH_IMU ok
<- PH_W ok
<- PH_MODE ok
<- PH_ON ok
<- T_STOP ok
<- PH_OFF ok
<- Smooth servo 0 -> 90.00
<- Smooth servo 1 -> 90.00
<- Smooth servo 2 -> 90.00


## Control manual

También se puede lanzar todo desde aquí sin la gráfica.

In [ ]:
# reproducir()                  # ir al inicio + trayectoria completa en segundo plano
# detener()
# a_90()                        # detener y todos los servos a 90°
# eris.enviar("CAL")            # nueva pose neutra de los IMUs
# eris.enviar("T_ST")           # estado de la cola: underruns, rechazados...
# eris.enviar("Y 0 90")         # servo 0 a 90° suave
# eris.enviar("X 0 90")         # servo 0 a 90° inmediato
eris.enviar("T_ST")

## Análisis: IMUs vs. trayectoria comandada

Relaciona cada articulación medida con una columna del CSV. Como los IMUs miden respecto
a la pose neutra y la trayectoria es absoluta, se comparan quitando la media de cada señal.

In [14]:
COMPARAR = {a: f"{PIERNA}_{a}_joint" for a in ARTICULACIONES}   # IMU de cada eslabón vs. su columna

eris.flush()
reg = pd.read_csv(ARCHIVO_CSV)
activo = reg[reg["traj_idx"] >= 1].copy()
if activo.empty:
    print("Todavía no hay datos grabados durante una trayectoria")
else:
    articulares = np.tile(tray[list(COMPARAR.values())].to_numpy(), (REPETICIONES, 1))
    punto = (activo["traj_idx"].to_numpy() - 1).clip(0, len(articulares) - 1)

    fig, axs = plt.subplots(len(COMPARAR), 1, sharex=True, figsize=(11, 8))
    for ax, (imu, col), j in zip(axs, COMPARAR.items(), range(len(COMPARAR))):
        medido = activo[imu].to_numpy()
        comando = articulares[punto, j]
        ax.plot(activo["t_s"], medido - medido.mean(), label=f"IMU {imu}")
        ax.plot(activo["t_s"], comando - comando.mean(), "--", label=f"comando {col}")
        ax.set_ylabel("Δ ángulo (°)")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)
    axs[-1].set_xlabel("Tiempo ESP32 (s)")
    fig.suptitle(ARCHIVO_CSV)
    plt.show()

In [24]:
eris.cerrar()
print("Puerto cerrado")

Puerto cerrado
